# What Should a Latent Message Preserve?

Cross-field idea: **rate-distortion theory with side information** (Wyner-Ziv / Slepian-Wolf,
information theory). If a decoder already has side information $Y$ correlated with the source
$X$, the encoder only needs to transmit the *conditional* information $H(X\mid Y)$, not $H(X)$.
Reconstructing $X$ faithfully wastes rate on what $Y$ already implies.

Source: Cover & Thomas, *Elements of Information Theory*, ch. 15; A. Wyner and J. Ziv,
"The rate-distortion function for source coding with side information at the decoder,"
IEEE Trans. Inform. Theory, 22(1), 1976.

**Mapping to LLM-agent latent handoff** (exact where noted, analogy otherwise):

| info-theory object | latent-handoff counterpart | status |
|---|---|---|
| source X | worker's full-context representation | exact object |
| side information Y | receiver's own context / priors | analogy (priors aren't a clean r.v.) |
| rate R | message budget k | exact |
| distortion D | downstream task loss | reframed (not reconstruction error) |
| decoder | receiver probe | exact |

**Falsifiable claim.** At a fixed budget $k$, selecting the message to carry information that
is *surprising to the receiver* (the residual of $X$ given $Y$) — optionally intersected with
*task relevance* — yields higher downstream accuracy than *reconstruction-based* selection
(top-variance directions of $X$).
- Supported if: SURPRISE / SURP+TASK > RECON at matched $k$.
- Against if: RECON $\ge$ SURPRISE at matched $k$.

This notebook (1) validates the experimental design on a synthetic generator with a known
ground truth, then (2) builds real hidden-state tensors from a small encoder and reruns the
same comparison on them.

## 1. Selection strategies and decoder (from `select_experiment.py`)

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, ".")
from select_experiment import (synthetic, load_real, zscore, residual_after_Y,
                                topk_pca_dirs, topk_task_dirs, logreg_fit, logreg_acc, run)
print("loaded selection experiment code")

loaded selection experiment code


Four selectors, all picking $k$ linear features, evaluated with the *same* logistic-regression
decoder that always sees the receiver's side info $Y$ plus the $k$-dim message:

- **RECON** — top-$k$ principal directions of $X$ (best reconstruction of the worker's state)
- **SURPRISE** — top-$k$ principal directions of the residual of $X$ after linearly predicting
  it from $Y$ (maximum conditional information, i.e. what $Y$ *cannot* already tell the receiver)
- **TASK** — $k$ features of $X$ most correlated with the label
- **SURP+TASK** — task-relevant features of the residual (conditional *and* relevant)

Baselines: `Y_only` (no message at all) and `Y+full_X` (unlimited budget) bracket the
achievable range.

## 2. Design validation on synthetic, ground-truth-controlled data

In [2]:
def sweep(X_Y_labels_fn, k, seeds, real=False):
    agg = {}
    for s in range(seeds):
        X, Y, labels = X_Y_labels_fn(s)
        res = run(X, Y, labels, k, seed=s)
        for name, acc in res.items():
            agg.setdefault(name, []).append(acc)
    order = ["Y_only(no msg)", "RECON", "TASK", "SURPRISE", "SURP+TASK", "Y+full_X(no budget)"]
    return {name: (np.mean(agg[name]), np.std(agg[name])) for name in order}

synthetic_results = sweep(lambda s: synthetic(seed=s), k=3, seeds=5)
for name, (m, sd) in synthetic_results.items():
    print(f"{name:22s} acc = {m:.4f} +/- {sd:.4f}")

Y_only(no msg)         acc = 0.6812 +/- 0.0093
RECON                  acc = 0.7107 +/- 0.0184
TASK                   acc = 0.7610 +/- 0.0284
SURPRISE               acc = 0.8162 +/- 0.0298
SURP+TASK              acc = 0.7803 +/- 0.0199
Y+full_X(no budget)    acc = 0.8947 +/- 0.0132


The synthetic generator deliberately loads the high-variance directions of $X$ onto a
receiver-known, label-irrelevant nuisance — the trap reconstruction-based selection falls
into. SURPRISE beating RECON here is the sanity check that the experimental design actually
measures what it claims to before touching any real model.

## 3. Real hidden states: building the saved tensors

Worker = full-sentence pooled hidden state; receiver side information = pooled hidden state of the first half of the sentence only (its side channel); label = SST-2 sentiment. Produced by `build_tensors.py` (`bert-tiny`, n=3000) and saved as `sst2_X.npy` / `sst2_Y.npy` / `sst2_labels.npy` — the saved hidden-state tensors submitted alongside this notebook.

In [3]:
X, Y, labels = load_real("sst2")
print("X (worker, full sentence):", X.shape)
print("Y (receiver side info, first half):", Y.shape)
print("labels:", labels.shape, "positive rate:", labels.mean())

X (worker, full sentence): (3000, 128)
Y (receiver side info, first half): (3000, 128)
labels: (3000,) positive rate: 0.5573333333333333


## 4. Same comparison on real hidden states

In [4]:
real_results = sweep(lambda s: (X, Y, labels), k=8, seeds=5)
for name, (m, sd) in real_results.items():
    print(f"{name:22s} acc = {m:.4f} +/- {sd:.4f}")

Y_only(no msg)         acc = 0.6589 +/- 0.0070
RECON                  acc = 0.6624 +/- 0.0080
TASK                   acc = 0.6896 +/- 0.0064
SURPRISE               acc = 0.7004 +/- 0.0114
SURP+TASK              acc = 0.6962 +/- 0.0047
Y+full_X(no budget)    acc = 0.6827 +/- 0.0069


In [5]:
names = list(real_results.keys())
means = [real_results[n][0] for n in names]
stds = [real_results[n][1] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, results, title in [(axes[0], synthetic_results, "Synthetic (k=3)"),
                            (axes[1], real_results, "Real SST-2 hidden states (k=8)")]:
    ns = list(results.keys())
    ms = [results[n][0] for n in ns]
    ss = [results[n][1] for n in ns]
    ax.bar(range(len(ns)), ms, yerr=ss, capsize=3)
    ax.set_xticks(range(len(ns)))
    ax.set_xticklabels(ns, rotation=45, ha="right")
    ax.set_ylabel("downstream accuracy")
    ax.set_title(title)
fig.tight_layout()
fig.savefig("selection_results.png", dpi=130)
print("saved selection_results.png")

saved selection_results.png


## 5. Interpretation

**Does it support the claim?** Yes, on both synthetic and real data: at matched budget,
SURPRISE and SURP+TASK both beat RECON. On the real SST-2 hidden states the effect is smaller
than the (ground-truth-engineered) synthetic gap, as expected, but the ordering
SURPRISE > SURP+TASK > TASK > RECON > Y_only holds.

**One alternative explanation.** On the synthetic data, pure SURPRISE beats SURP+TASK because
the novel signal already dominates residual variance by construction — "surprise" and
"task-relevant" coincide because the generator was built that way. On the real data SURPRISE
still edges out SURP+TASK, which suggests the residual's top-variance directions are already
fairly well aligned with sentiment-relevant content in this encoder, so the extra
task-correlation filter mostly trims noise rather than adding new signal — but a different
encoder or a harder task could easily flip this.

**Where the connection breaks down.** The linear residual under-measures what the receiver
actually knows: a strong nonlinear receiver could reconstruct more from $Y$ than a linear
regression removes, so "surprise" as computed here is only an *upper bound* on what must be
sent. Likewise, treating "the receiver's own context and model priors" as a side-information
random variable is an analogy, not an exact correspondence — there's no clean distribution
backing it.

**A genuinely surprising real-data result.** `Y+full_X` (the "no budget" bracket, which should
be an upper bound) scores *lower* than SURPRISE/SURP+TASK/TASK on the real tensors, unlike on
the synthetic data. The most likely explanation is that handing a simple L2-regularized
logistic-regression probe the full 128-dim $X$ on top of $Y$, with a 70/30 train/test split
of 3,000 examples, adds enough noisy/collinear dimensions to hurt a linear decoder — an
artifact of decoder capacity and sample size, not evidence that unlimited budget is worse in
principle. It's a useful caution: the "no-budget" bracket is only a genuine upper bound when
the decoder can actually make use of the extra information.

**Follow-up experiment.** Replace the linear residual with a learned (nonlinear) predictor of
$X$ from $Y$ to define "surprise", sweep $k$ to trace an operational rate-distortion curve per
selector, and vary receiver strength to test whether a *stronger* receiver needs a *smaller*
message — the prediction implied by the Wyner-Ziv framing but not tested here.